# Workout Plan Generator - Model Testing Notebook

This notebook tests the fine-tuned Flan-T5 model for generating personalized workout plans.

## What we'll test:
1. ✅ Model loading and inference
2. ✅ JSON output validity
3. ✅ Different fitness levels (Beginner, Intermediate, Advanced)
4. ✅ Different goals (Strength, Muscle, Weight Loss)
5. ✅ Equipment constraints
6. ✅ Injury restrictions
7. ✅ Latency benchmarks
8. ✅ Plan quality evaluation

**Prerequisites**: 
- Model trained and saved in `./models/workout-generator-v1/`
- Required packages installed (see requirements.txt)

## 1. Setup & Imports

In [ ]:
# Install required packages (run once)
!pip install -q torch transformers peft datasets accelerate

In [ ]:
import os
import json
import time
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import PeftModel
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  Using CPU (inference will be slower)")

## 2. Load Trained Model

In [ ]:
# Model paths
MODEL_PATH = "./models/workout-generator-v1"
LORA_ADAPTER_PATH = "./models/workout-generator-v1/lora_adapter"

# Check if model exists
if not os.path.exists(MODEL_PATH):
    print("❌ Model not found!")
    print(f"   Expected location: {MODEL_PATH}")
    print("   Run training first: python train.py --generate-data 5000 --epochs 5")
else:
    print(f"✅ Model found at {MODEL_PATH}")

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

print("Loading base model...")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=dtype,
    device_map="auto"
)

# Load LoRA adapter if exists
if os.path.exists(LORA_ADAPTER_PATH):
    print("Loading LoRA adapter...")
    model = PeftModel.from_pretrained(model, LORA_ADAPTER_PATH)
    print("✅ LoRA adapter loaded")
else:
    print("⚠️  LoRA adapter not found, using base model only")

model.eval()
print(f"\n✅ Model loaded successfully!")
print(f"   Device: {device}")
print(f"   Parameters: {model.num_parameters():,}")

## 3. Helper Functions

In [ ]:
def generate_workout_plan(
    prompt: str,
    max_length: int = 2048,
    num_beams: int = 4,
    verbose: bool = True
) -> Dict:
    """
    Generate workout plan from prompt
    
    Returns:
        {
            'prompt': str,
            'generated_text': str,
            'plan': dict (if valid JSON),
            'is_valid_json': bool,
            'latency_ms': int,
            'error': str (if any)
        }
    """
    start_time = time.time()
    
    # Tokenize input
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            num_beams=num_beams,
            do_sample=False,
            early_stopping=True
        )
    
    # Decode
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    latency_ms = int((time.time() - start_time) * 1000)
    
    result = {
        'prompt': prompt,
        'generated_text': generated_text,
        'latency_ms': latency_ms
    }
    
    # Try to parse as JSON
    try:
        plan = json.loads(generated_text)
        result['plan'] = plan
        result['is_valid_json'] = True
        result['error'] = None
    except (json.JSONDecodeError, ValueError) as e:
        result['plan'] = None
        result['is_valid_json'] = False
        result['error'] = str(e)
    
    if verbose:
        print(f"⏱️  Latency: {latency_ms}ms")
        print(f"{'✅' if result['is_valid_json'] else '❌'} Valid JSON: {result['is_valid_json']}")
    
    return result


def print_workout_plan(plan: Dict) -> None:
    """Pretty print workout plan"""
    print("\n" + "="*70)
    print(f"📋 {plan.get('plan_name', 'Workout Plan')}")
    print("="*70)
    
    if 'description' in plan:
        print(f"\n{plan['description']}")
    
    print(f"\n📊 Overview:")
    print(f"   Difficulty: {plan.get('difficulty', 'N/A')}")
    print(f"   Goal: {plan.get('goal', 'N/A')}")
    print(f"   Days/Week: {plan.get('days_per_week', 'N/A')}")
    print(f"   Duration: {plan.get('duration_weeks', 'N/A')} weeks")
    
    if 'days' in plan:
        for day in plan['days']:
            print(f"\n📅 Day {day.get('day_number', '?')} - {day.get('day_name', 'Workout Day')}")
            print("   " + "-"*65)
            
            for i, exercise in enumerate(day.get('exercises', []), 1):
                print(f"   {i}. {exercise.get('exercise_name', 'Unknown')}")
                print(f"      Sets: {exercise.get('sets', '?')} | "
                      f"Reps: {exercise.get('reps', '?')} | "
                      f"Rest: {exercise.get('rest_seconds', '?')}s")
                if 'notes' in exercise:
                    print(f"      💡 {exercise['notes']}")
    
    if 'progressive_overload' in plan:
        print(f"\n📈 Progressive Overload:")
        for phase, instruction in plan['progressive_overload'].items():
            print(f"   {phase}: {instruction}")
    
    if 'metadata' in plan:
        print(f"\n📊 Metadata:")
        for key, value in plan['metadata'].items():
            print(f"   {key}: {value}")
    
    print("\n" + "="*70 + "\n")


def validate_plan_structure(plan: Dict) -> Dict:
    """
    Validate workout plan structure
    
    Returns:
        {
            'is_valid': bool,
            'errors': List[str],
            'warnings': List[str]
        }
    """
    errors = []
    warnings = []
    
    # Required fields
    required_fields = ['plan_name', 'days_per_week', 'days']
    for field in required_fields:
        if field not in plan:
            errors.append(f"Missing required field: {field}")
    
    # Validate days
    if 'days' in plan:
        if not isinstance(plan['days'], list):
            errors.append("'days' must be a list")
        elif len(plan['days']) == 0:
            errors.append("No workout days provided")
        else:
            # Validate each day
            for i, day in enumerate(plan['days']):
                if 'exercises' not in day:
                    errors.append(f"Day {i+1} missing 'exercises'")
                elif len(day['exercises']) == 0:
                    warnings.append(f"Day {i+1} has no exercises")
                elif len(day['exercises']) > 10:
                    warnings.append(f"Day {i+1} has {len(day['exercises'])} exercises (might be too many)")
    
    return {
        'is_valid': len(errors) == 0,
        'errors': errors,
        'warnings': warnings
    }

print("✅ Helper functions loaded")

## 4. Basic Test - Single Generation

In [ ]:
# Simple test
test_prompt = "Generate a 4-day workout plan for intermediate lifter, goal is muscle gain, has dumbbells and barbell."

print(f"📝 Prompt: {test_prompt}\n")
result = generate_workout_plan(test_prompt, verbose=True)

if result['is_valid_json']:
    print_workout_plan(result['plan'])
    
    # Validate structure
    validation = validate_plan_structure(result['plan'])
    print(f"\n{'✅' if validation['is_valid'] else '❌'} Structure Validation: {validation['is_valid']}")
    if validation['errors']:
        print(f"   Errors: {validation['errors']}")
    if validation['warnings']:
        print(f"   Warnings: {validation['warnings']}")
else:
    print(f"\n❌ Failed to generate valid JSON")
    print(f"   Error: {result['error']}")
    print(f"\n   Raw output (first 500 chars):\n{result['generated_text'][:500]}...")

## 5. Test Different Scenarios

In [ ]:
# Define test cases
test_cases = [
    {
        "name": "Beginner - Full Body",
        "prompt": "Generate a 3-day workout plan for beginner, goal is muscle gain, has access to dumbbells only."
    },
    {
        "name": "Intermediate - Upper/Lower Split",
        "prompt": "Generate a 4-day workout plan for intermediate lifter, goal is strength, has full gym access."
    },
    {
        "name": "Advanced - PPL",
        "prompt": "Generate a 6-day workout plan for advanced lifter, goal is muscle hypertrophy, has barbell, dumbbells, cables."
    },
    {
        "name": "Weight Loss - Bodyweight",
        "prompt": "Generate a 5-day workout plan for intermediate, goal is weight loss, bodyweight exercises only."
    },
    {
        "name": "With Injury Restriction",
        "prompt": "Generate a 4-day workout plan for intermediate, goal is muscle gain, has full gym, avoid shoulder exercises due to injury."
    },
    {
        "name": "Strength Focus",
        "prompt": "Generate a 3-day workout plan for advanced powerlifter, goal is strength, has barbell and bench."
    }
]

print(f"Running {len(test_cases)} test cases...\n")

In [ ]:
# Run all test cases
test_results = []

for i, test_case in enumerate(test_cases, 1):
    print(f"\n{'='*70}")
    print(f"Test {i}/{len(test_cases)}: {test_case['name']}")
    print(f"{'='*70}")
    print(f"Prompt: {test_case['prompt']}\n")
    
    result = generate_workout_plan(test_case['prompt'], verbose=True)
    result['test_name'] = test_case['name']
    test_results.append(result)
    
    if result['is_valid_json']:
        validation = validate_plan_structure(result['plan'])
        result['validation'] = validation
        
        # Print summary
        plan = result['plan']
        total_exercises = sum(len(day.get('exercises', [])) for day in plan.get('days', []))
        
        print(f"   Plan: {plan.get('plan_name', 'N/A')}")
        print(f"   Days: {len(plan.get('days', []))}")
        print(f"   Total Exercises: {total_exercises}")
        print(f"   Structure Valid: {validation['is_valid']}")
    else:
        print(f"   ❌ JSON parsing failed")

print(f"\n\n✅ All {len(test_cases)} test cases completed!")

## 6. Results Analysis

In [ ]:
# Compute metrics
total_tests = len(test_results)
valid_json_count = sum(1 for r in test_results if r['is_valid_json'])
valid_structure_count = sum(
    1 for r in test_results 
    if r.get('validation', {}).get('is_valid', False)
)
avg_latency = sum(r['latency_ms'] for r in test_results) / total_tests

print("📊 Test Results Summary")
print("="*70)
print(f"Total Tests: {total_tests}")
print(f"Valid JSON: {valid_json_count}/{total_tests} ({valid_json_count/total_tests*100:.1f}%)")
print(f"Valid Structure: {valid_structure_count}/{total_tests} ({valid_structure_count/total_tests*100:.1f}%)")
print(f"Average Latency: {avg_latency:.0f}ms")
print(f"Min Latency: {min(r['latency_ms'] for r in test_results)}ms")
print(f"Max Latency: {max(r['latency_ms'] for r in test_results)}ms")
print("="*70)

In [ ]:
# Create results DataFrame
results_df = pd.DataFrame([
    {
        'Test': r['test_name'],
        'Valid JSON': r['is_valid_json'],
        'Valid Structure': r.get('validation', {}).get('is_valid', False),
        'Latency (ms)': r['latency_ms'],
        'Days': len(r.get('plan', {}).get('days', [])) if r['is_valid_json'] else 0,
        'Total Exercises': sum(
            len(day.get('exercises', [])) 
            for day in r.get('plan', {}).get('days', [])
        ) if r['is_valid_json'] else 0
    }
    for r in test_results
])

print("\n📋 Detailed Results:")
print(results_df.to_string(index=False))

In [ ]:
# Visualize latency
plt.figure(figsize=(12, 5))

# Latency by test
plt.subplot(1, 2, 1)
plt.bar(range(len(results_df)), results_df['Latency (ms)'])
plt.axhline(y=2000, color='r', linestyle='--', label='2s Target')
plt.xlabel('Test Case')
plt.ylabel('Latency (ms)')
plt.title('Inference Latency by Test Case')
plt.legend()
plt.xticks(range(len(results_df)), [f"T{i+1}" for i in range(len(results_df))])

# Success rate
plt.subplot(1, 2, 2)
success_metrics = [
    ('Valid JSON', results_df['Valid JSON'].sum()),
    ('Valid Structure', results_df['Valid Structure'].sum())
]
labels, values = zip(*success_metrics)
plt.bar(labels, values)
plt.axhline(y=total_tests, color='g', linestyle='--', label='All Tests')
plt.ylabel('Count')
plt.title('Validation Success Rate')
plt.ylim(0, total_tests + 1)
plt.legend()

plt.tight_layout()
plt.show()

print(f"\n📈 Target: <2000ms latency, >95% JSON validity")
print(f"   Current: {avg_latency:.0f}ms latency, {valid_json_count/total_tests*100:.1f}% JSON validity")
if avg_latency < 2000 and valid_json_count/total_tests >= 0.95:
    print("   ✅ MEETS PRODUCTION TARGETS")
else:
    print("   ⚠️  Needs improvement")

## 7. Detailed Plan Inspection

In [ ]:
# Pick a successful test to inspect in detail
successful_results = [r for r in test_results if r['is_valid_json']]

if successful_results:
    sample_result = successful_results[0]
    print(f"Inspecting: {sample_result['test_name']}\n")
    print_workout_plan(sample_result['plan'])
    
    # Show raw JSON
    print("\n📄 Raw JSON Output:")
    print("="*70)
    print(json.dumps(sample_result['plan'], indent=2)[:1000] + "...\n")
else:
    print("❌ No successful tests to inspect")

## 8. Stress Test - Latency Benchmark

In [ ]:
# Run multiple inferences to measure consistency
print("Running stress test (10 iterations)...\n")

stress_test_prompt = "Generate a 4-day workout plan for intermediate, goal is muscle gain, has full gym."
latencies = []

for i in range(10):
    result = generate_workout_plan(stress_test_prompt, verbose=False)
    latencies.append(result['latency_ms'])
    print(f"  Iteration {i+1}: {result['latency_ms']}ms | Valid: {result['is_valid_json']}")

print(f"\n📊 Latency Statistics:")
print(f"   Mean: {sum(latencies)/len(latencies):.0f}ms")
print(f"   Median: {sorted(latencies)[len(latencies)//2]:.0f}ms")
print(f"   Min: {min(latencies)}ms")
print(f"   Max: {max(latencies)}ms")
print(f"   Std Dev: {pd.Series(latencies).std():.0f}ms")

# Plot distribution
plt.figure(figsize=(10, 4))
plt.hist(latencies, bins=10, edgecolor='black')
plt.axvline(x=2000, color='r', linestyle='--', label='2s Target')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.title('Latency Distribution (10 iterations)')
plt.legend()
plt.show()

## 9. Interactive Testing

Try your own prompts!

In [ ]:
# Custom prompt
custom_prompt = "Generate a 5-day workout plan for advanced athlete, goal is endurance, has kettlebells and pull-up bar."

print(f"📝 Your Prompt: {custom_prompt}\n")
result = generate_workout_plan(custom_prompt)

if result['is_valid_json']:
    print_workout_plan(result['plan'])
else:
    print(f"❌ Failed to generate valid plan")
    print(f"   Error: {result['error']}")

## 10. Export Test Results

In [ ]:
# Save results to JSON
output_file = "test_results.json"

test_summary = {
    "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
    "model_path": MODEL_PATH,
    "device": device,
    "metrics": {
        "total_tests": total_tests,
        "valid_json_count": valid_json_count,
        "valid_json_rate": valid_json_count / total_tests,
        "valid_structure_count": valid_structure_count,
        "avg_latency_ms": avg_latency,
        "min_latency_ms": min(r['latency_ms'] for r in test_results),
        "max_latency_ms": max(r['latency_ms'] for r in test_results)
    },
    "test_results": [
        {
            "test_name": r['test_name'],
            "prompt": r['prompt'],
            "is_valid_json": r['is_valid_json'],
            "latency_ms": r['latency_ms'],
            "plan": r.get('plan'),
            "error": r.get('error')
        }
        for r in test_results
    ]
}

with open(output_file, 'w') as f:
    json.dump(test_summary, f, indent=2)

print(f"✅ Test results saved to {output_file}")
print(f"\n📊 Summary:")
print(json.dumps(test_summary['metrics'], indent=2))

## 11. Production Readiness Checklist

Evaluate if model is ready for deployment:

In [ ]:
# Production criteria
criteria = {
    "JSON Validity >95%": (valid_json_count / total_tests) >= 0.95,
    "P95 Latency <2s": avg_latency < 2000,
    "No Critical Errors": all(r.get('validation', {}).get('is_valid', True) for r in test_results if r['is_valid_json']),
    "Handles All Scenarios": total_tests >= 6,
}

print("\n🎯 Production Readiness Checklist")
print("="*70)

all_pass = True
for criterion, passed in criteria.items():
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{status}: {criterion}")
    if not passed:
        all_pass = False

print("="*70)

if all_pass:
    print("\n🚀 MODEL IS PRODUCTION READY!")
    print("\nNext steps:")
    print("1. Deploy to FastAPI service (see IMPLEMENTATION_SPEC.md Section 5)")
    print("2. Integrate with C# backend (see IMPLEMENTATION_SPEC.md Section 6)")
    print("3. Setup monitoring (Prometheus + Grafana)")
    print("4. Deploy as canary release (5% traffic)")
else:
    print("\n⚠️  MODEL NEEDS IMPROVEMENT")
    print("\nRecommended actions:")
    if (valid_json_count / total_tests) < 0.95:
        print("- Increase training data (need 5K+ samples)")
        print("- Train for more epochs (5-10)")
        print("- Adjust generation parameters (beam search)")
    if avg_latency >= 2000:
        print("- Use GPU for inference")
        print("- Reduce max_output_length")
        print("- Implement model quantization")
    print("\nRe-run this notebook after improvements.")

## Summary

This notebook tested the fine-tuned Flan-T5 model across multiple scenarios.

**Key Metrics**:
- JSON Validity Rate
- Inference Latency
- Plan Structure Quality
- Handling of Different Scenarios

**Next Steps**:
1. If tests pass → Deploy to production
2. If tests fail → Retrain with more data/epochs
3. Monitor performance in production
4. Collect user feedback for continuous improvement

See [IMPLEMENTATION_SPEC.md](./IMPLEMENTATION_SPEC.md) for deployment details.